In [1]:
'''Functions'''

%matplotlib inline
#import b2plot as bp
import plothist as ph
import numpy as np
import pandas as pd
import scipy as sci
import sympy as sym
import statistics as stat
import random as rng
import sklearn as skl
import xgboost as xgb
import seaborn as sns
import itertools as ite
#import time
import math
import pdg
#import numba as nb
import boost_histogram as bh
import awkward as ak
#import uproot
import vector as vec
import matplotlib_inline
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
#------------------

#import matplotlib.animation as manim
#from matplotlib.ticker import AutoMinorLocator
#import cartopy.crs as ccr
#from cartopy.io.img_tiles import GoogleTiles
#import timeit
#import Ipython.display
#from IPython.display import display
from mpl_toolkits import mplot3d
from matplotlib.ticker import AutoMinorLocator, MaxNLocator, FuncFormatter, ScalarFormatter, LogLocator, LogFormatter
from itertools import combinations
from scipy.optimize import minimize
from IPython.display import display, Math
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
#from numba import njit
#---------
#import json
import os
#import gc
import sys
#----------

pd.options.display.max_columns = None

def lorentz_boost(four_vec, beta): #lorents boost for any frame
    beta = np.asarray(beta).reshape(3,)
    beta_sqr = np.dot(beta, beta)
    if beta_sqr == 0.0:
        return four_vec
    gamma = (1.0 - beta_sqr)**(-0.5)
    gammafrac = (gamma**2 / (1+gamma))
    bx, by, bz = beta

    loren = np.array([
        [gamma, -gamma*bx, -gamma*by, -gamma*bz],
        [-gamma*bx, 1+gammafrac*bx*bx, gammafrac*bx*by, gammafrac*bx*bz],
        [-gamma*by, gammafrac*by*bx, 1+gammafrac*by*by, gammafrac*by*bz],
        [-gamma*bz, gammafrac*bz*bx, gammafrac*bz*by, 1+gammafrac*bz*bz]
    ])
    return np.matmul(loren,four_vec)

def isotropic_direction(): #Return a random uniform selection of angles for spherical coordinates
    cos_theta = np.random.uniform(-1, 1)
    sin_theta = np.sqrt(1 - cos_theta**2)
    phi = np.random.uniform(0, 2 * np.pi)
    return np.array([sin_theta * np.cos(phi), sin_theta * np.sin(phi), cos_theta])

def four_vect(E, p3): #Given E and 3vec of momentum, make 4vec. 
    return np.array([E, *p3])

def sph_crt(phi, theta): #To cartesian from polar
    z = np.cos(theta)
    cos_phi = np.cos(phi)
    sin_phi = np.sin(phi)
    sin_theta = np.sin(theta)
    x = sin_theta * cos_phi
    y = sin_theta * sin_phi
    return x,y,z

def crt_sph(x,y,z): #to polar from cartesian
    theta = np.arccos(z)
    phi = np.arctan(y/x)
    return phi, theta

def calc_p4(E, p, costh, phi):  #Calculate the four vector of a particle (E,px,py,pz), Z axis is beam axis, phi is azimuthal
    sinth = np.sqrt(1 - costh**2)
    px = p * sinth * np.cos(phi)
    py = p * sinth * np.sin(phi)
    pz = p * costh
    return np.array([E, px, py, pz])

def calc_rapid(E, p, costh): #Rapidity
    return 0.5*np.ln( (E + p*costh) / (E - p*costh) )

def calc_prapid(costh): #pseudorapidity, usually eta
    return np.arctanh(costh)

def calc_betagamma(E,p): #Returns a tuple of beta v/c, and gamma. C = 1. 
    return p/E, E / np.sqrt(E**2-p**2)

def calc_pt(p, costh, phi): #Calculate transverse momentum
    sinth = np.sqrt(1 - costh**2)
    return ((p * sinth * np.cos(phi))**2 + (p * sinth * np.sin(phi))**2)**0.5

def calc_pl(p,costh): #calculate along beam axis (longitude) momentum
    return p * costh

def calc_InvM2(E,p):
    return E**2-p**2

def calc_flight_dist(dr,dz):
    return np.sqrt(dr**2+dz**2)

def calc_eff(passed_mask):
    return np.sum(passed_mask) / len(passed_mask)

def is_true_KS(ks_mcPDG, pi1_mother_PDG, pi2_mother_PDG):
    """Check if candidate is true K_S^0 based on MC truth."""
    return (ks_mcPDG == 310) & (pi1_mother_PDG == 310) & (pi2_mother_PDG == 310)


def plot_sim(datas, particle, quantity, src_names, bin_c=100, ax=None, src_count=3, scale_weight=1.0, bin_edges=None, err=False): 
    '''
    Args
    -> datas, list of data
    -> particle, name as a string, ie (D0SL)
    -> quantity, as a string, ie (InvM)
    -> d_id, filtering id, integer
    -> src_names, list of decay source names (in order of data list)
    -> bin_c, bin number
    -> ax, axis of subplot none by default for a single plot. axs[i] object otherwise
    -> src_count, count of sources to plot, 3 default
    -> scale_weight, normalization for all MC samples combined for decay ID and every entry
    -> bin_edges, for custom fitting
    -> err, for including attempt at source uncertainty
    -> filtarg, sorting filter or column (please use only ones that take whole numbers), decayModeID by default
    Future iterations will reduce parameter needs, improve the filtering and collecting algorithm, adjust layouts, and add more functionality
    '''
    phdl = particle+'_'+quantity
    
    data_list = []
    weight_list = []
    label_list = []
    
    for i in range(src_count):
        try:
            data = datas[i].query(f'{particle}_{filtarg}=={d_id}')[phdl] #Filter to quantities on filtering arg
        except:
            data = datas[i].query(f'{filtarg}=={d_id}')[phdl] #Yes, this is the laziest fix in the world for ncandidates

        data_list.append(data) #Grab all filtered ID data of interest via the loop
        weight_list.append(np.full(len(data),scale_weight)) #To prevent issues with non uniformity
        label_list.append(f'{filtarg}: {d_id}\nSource: {src_names[i]}') #Grab labels
    
    if ax is not None and not err:
        ax.hist(data_list, bins=bin_edges, stacked=True,weights=weight_list,label=label_list,edgecolor='black',align='mid')  
    elif ax is not None:
        ax.hist(data_list, bins=bin_edges, stacked=True,weights=weight_list,label=label_list,edgecolor='black')
        counts, edges, __ = ax.hist(data_list, bins=bin_edges, stacked=False, weights=weight_list, alpha=0)
        top_y_values = np.sum(counts, axis=0)
        centers = 0.5 * (edges[:-1] + edges[1:])
        step_size = bin_edges[1] - bin_edges[0]
        errors = np.sqrt(top_y_values)
        ax.bar(centers,height=2*errors,bottom=top_y_values - errors,alpha=0.8, #This was a pain to get working, have to draw invisible histogram to avoid the stacking
            color='none',edgecolor='black',hatch='////',width=step_size,label='Sim Uncertainty')
    else:
        plt.hist(data_list, bins=bin_edges,stacked=True,weights=weight_list,label=label_list,edgecolor='black')    

In [2]:
#mu+-, e+-, K+-, Pi+-
#
# 17.22439940134783% of SIM when using notrks and minE initial selections.

#Selection on events to get the four main B+ to Tr+ K0s modes  28961, specifically rest of the events
#Note: Trk charge is always -1 * esl charge. Esl charge always matched PID. Only -1 and 1. 


# These turned out to work great for tracks. 
# def sel_tau(df): Terrible, selecting taus is just not happening by trkk0s cuts, try other vars
#     return df[df['trkK0S_InvM'].between(1.72, 1.82)] #1.77685, 1.77687 Or make loose
#   #yet worked here. Strange. Make between just to be consistent

# 0.105658    13315
# 0.000511    12114
# 0.139570     9032
# 0.493677     3953
#Also check to make sure trk type and PID value correlate
#Correlate look for PDG PID and trk INVM

# eSL_charge
#  1.0    19408
# -1.0    19006
# df_chg = assign_sig(df_chg)
# df_chg
# Name: count, dtype: int64 
"""notes"""

def assign_sig2(df,mode1 = 'Bplus', mode2 = 'Bminus'): #for sorting decay toplogies
    df = df.copy()
    df['signal'+mode1] = df['a'+mode1+'Mode'].where(df['eSL_charge'] < 0, other=pd.NA)
    #df['McMatch_eSL'] = df['eSL_PDG'] == df['eSL_mcPDG']
    df['signal'+mode2] = df['a'+mode2+'Mode'].where(df['eSL_charge'] > 0, other=pd.NA)
    return df
# def assign_sig1(df):
#     df = df.copy() #THIS FUNCTION currently is WRONG. I DOUBT the same B0 and B+ numbers correspond only changed by the Parent
#     df.loc[(df['eSL_charge'] < 0) & (df['aD0Mode']    != -99.0), 'signalBplus'] = df['aD0Mode']
#     df.loc[(df['eSL_charge'] < 0) & (df['aDplusMode'] != -99.0), 'signalBplus'] = df['aDplusMode']
#     df.loc[(df['eSL_charge'] < 0) & (df['aB0Mode']    != -99.0), 'signalBplus'] = df['aB0Mode']
#     df.loc[(df['eSL_charge'] < 0) & (df['aBplusMode'] != -99.0), 'signalBplus'] = df['aBplusMode']

#     df.loc[(df['eSL_charge'] > 0) & (df['aD0Mode']     != -99.0), 'signalBminus'] = df['aDbar0Mode']
#     df.loc[(df['eSL_charge'] > 0) & (df['aDminusMode'] != -99.0), 'signalBminus'] = df['aDminusMode']
#     df.loc[(df['eSL_charge'] > 0) & (df['aBbar0Mode']  != -99.0), 'signalBminus'] = df['aBbar0Mode']
#     df.loc[(df['eSL_charge'] > 0) & (df['aBminusMode'] != -99.0), 'signalBminus'] = df['aBminusMode']
#     #Done in order of priority
#     df.loc[pd.isna(df['signalBplus']) & pd.isna(df['signalBminus']), 'signalBplus'] = -99
#     return df
    #My attempt to differ is signal side from tag side across all MC

def assign_sig(df):
    df = df.copy() 
    df.loc[(df['eSL_charge'] < 0) & (df['aD0Mode']    != -99.0), 'signalD0'] = df['aD0Mode']
    df.loc[(df['eSL_charge'] < 0) & (df['aDplusMode'] != -99.0), 'signalD+'] = df['aDplusMode']
    df.loc[(df['eSL_charge'] < 0) & (df['aB0Mode']    != -99.0), 'signalB0'] = df['aB0Mode']
    df.loc[(df['eSL_charge'] < 0) & (df['aBplusMode'] != -99.0), 'signalBplus'] = df['aBplusMode']

    df.loc[(df['eSL_charge'] > 0) & (df['aD0Mode']     != -99.0), 'signalD0-'] = df['aDbar0Mode']
    df.loc[(df['eSL_charge'] > 0) & (df['aDminusMode'] != -99.0), 'signalD+'] = df['aDminusMode']
    df.loc[(df['eSL_charge'] > 0) & (df['aBbar0Mode']  != -99.0), 'signalB0-'] = df['aBbar0Mode']
    df.loc[(df['eSL_charge'] > 0) & (df['aBminusMode'] != -99.0), 'signalBminus'] = df['aBminusMode']
    #Done in order of priority
    df.loc[df[['signalBplus', 'signalBminus', 'signalB0', 'signalB0-', 'signalD0', 'signalD0-', 'signalD+']].isna().all(axis=1), 'signalBplus'] = -99
#Make way to erase duplicate Signal Modes
    return df

def sel_no_chgtrks(df):
    df = df.copy()
    df = df.query('nROE_Ch == 0') #Mandatory
    return df #79116, 38414 48.554% of events, 69.3885449% of data from sel
    """I moved these up here, although I should have kept them with their plots to show the signal cut effect, will do so in future iterated cuts"""
# def sel_minEex(df):
#     df = df.copy()
#     return df.query('Eextra_ROE < 1.05') #14.86% of data after notrk and minE sel initial. 48.55% of second
#     #19813 from 0.75, 38414 from 1.05, so about ~7.6% if we use 0.75GeV
def sel_mu(df):
    df_t = df.copy()
    # return df[df['trk_InvM'].between(0.105657, 0.105659)] #weird glitch where if exactly equal ,despite what says in data. Wont get, use between. FP error... truncating?
    return df_t[(df_t['trk_PDG'] == 13) | (df_t['trk_PDG'] == -13)]
def sel_epos(df):
    df_t = df.copy()
    return df_t[(df_t['trk_PDG'] == 11) | (df_t['trk_PDG'] == -11)]
def sel_pichg(df):
    df_t = df.copy()
    return df_t[(df_t['trk_PDG'] == 211) | (df_t['trk_PDG'] == -211)]
def sel_kchg(df):
    df_t = df.copy()
    return df_t[(df_t['trk_PDG'] == 321) | (df_t['trk_PDG'] == -321)]

def sel_ksig(df):
    df_t = df.copy()
    return df.query('K0S_isSignal == 1.0')

def sel_copy(df):
    df_t = df.copy()
    return df_t

def plot_mc_with_err(ax, stacked_data, bins, weight=1.0):
    total_vals = np.concatenate(stacked_data)
    counts, edges = np.histogram(total_vals, bins=bins)
    counts = weight * counts
    errors = weight * np.sqrt(np.histogram(total_vals, bins=bins)[0])
    bin_centers = 0.5 * (edges[1:] + edges[:-1])
    ax.bar(bin_centers, 2 * errors, bottom=counts - errors, width=np.diff(bins), 
           hatch='////', fill=False, edgecolor='gray', linewidth=0.0, zorder=5)

In [3]:
#28.96% of simulation was charged semi leptonic decays

# {'semileptonic': 28.955732613566564,
#  'ew/leptonic/radiative/rare': 0.10433157717162804,
#  'hadronic': 4.327424671044094,
#  'unclassified': 18.69230835777253}

    # for dx in datasmc:
    #     sel = ((dx['nROE_Ch'] == 0) & (dx['Eextra_ROE'] < 1.05) & (np.abs(dx['cosThCM']) < 0.9) &
    #            (dx['M_ROE'] > 0) & (dx['M_ROE'] < 3.5) & (dx['nROE_ECL'] > 3) & (dx['cosK0Strk'] < 0.1) & (dx['nROE_ECL_loose'] > 5.0)
    #             & (dx['R2'] < 0.405) & (dx['cosTBTO'] < 0.9) & (dx['K0S_flightDistance'] > 0) & (dx['cosK0SD0SL'] > -0.8)
    #             & (dx['Eextra_ROE_loose'] > 0.803) & (dx['M_ROE_loose'] > 0.56) & (dx['M_ROE_loose'] < 1.11) & (dx['trkK0S_InvM'] < 4.67)
    #             & (dx['trkK0S_InvM'] > 0.67) & (dx['trkK0S_pCM'] < 2.33) & (dx['K0S_flightDistance'] > 0.14) & (dx['K0S_dr'] > 0.1) & (dx['trkK0S_ECM'] > 0.924))
    #     sigs.append(dx[sel & (dx['K0S_isSignal'] == 1.0)][var])
    #     bkgs.append(dx[sel & (dx['K0S_isSignal'] != 1.0)][var])
        
# sel = ((df_dat['nROE_Ch'] == 0) & (df_dat['Eextra_ROE'] < 1.05) & (np.abs(df_dat['cosThCM']) < 0.9) &
#        (df_dat['M_ROE'] > 0) & (df_dat['M_ROE'] < 3.5) & (df_dat['nROE_ECL'] > 3) & (df_dat['cosK0Strk'] < 0.1) & (df_dat['nROE_ECL_loose'] > 5.0)
#        & (df_dat['R2'] < 0.405) & (df_dat['cosTBTO'] < 0.9) & (df_dat['K0S_flightDistance'] > 0) & (df_dat['cosK0SD0SL'] > -0.8) 
#        & (df_dat['Eextra_ROE_loose'] > 0.803) & (df_dat['M_ROE_loose'] > 0.56) & (df_dat['M_ROE_loose'] < 1.11) & (df_dat['trkK0S_InvM'] < 4.67)
#        & (df_dat['trkK0S_InvM'] > 0.67) & (df_dat['trkK0S_pCM'] < 2.33) & (df_dat['K0S_flightDistance'] > 0.14) & (df_dat['K0S_dr'] > 0.1) & (df_dat['trkK0S_ECM'] > 0.924))
    # sel_dat = df_dat[sel]

def msk_tot_ini(df): #1.08% of MC. Likely need to loosen some latters and earliers, especially kinematic ranges, 1.17% of Data
    return ((df['nROE_Ch'] == 0) & (df['Eextra_ROE'] < 1.05) & (np.abs(df['cosThCM']) < 0.9) &
       (df['M_ROE'] > 0) & (df['M_ROE'] < 3.5) & (df['nROE_ECL'] > 3) & (df['cosK0Strk'] < 0.1) & (df['nROE_ECL_loose'] > 5.0)
       & (df['R2'] < 0.405) & (df['cosTBTO'] < 0.9) & (df['K0S_flightDistance'] > 0) & (df['cosK0SD0SL'] > -0.8) 
       & (df['Eextra_ROE_loose'] > 0.803) & (df['M_ROE_loose'] > 0.56) & (df['M_ROE_loose'] < 1.11) & (df['trkK0S_InvM'] < 4.67)
       & (df['trkK0S_InvM'] > 0.67) & (df['trkK0S_pCM'] < 2.33) & (df['K0S_flightDistance'] > 0.14) & (df['K0S_dr'] > 0.1) & (df['trkK0S_ECM'] > 0.924))

def sel_k0s_sig(df):
    return (df['K0S_isSignal'] == 1).copy()
def sel_trk_sig(df):
    return (df['trk_isSignal'] == 1).copy()

#66.21% pass as signal, 66.93% for mcPDG
#33.79% with 3.25 less from is 0
#90.31% trk is signal 
#17.76% independent from trkK0S_isSignalAcceptmissing
#17.95% from working BSL^^
#32.52% from D tag^^
#89.52% from lep tag^^
#29.76 from no chg trks in ROE
#10.08 from Gamma4s ^^
#60.35 contributes once, 
#13.06 contributes twice
#1.96 contributes 3 times #Nans not included!!!
#77.77% of rows have a K0S I think...
#102632,452687,125278,17.76 195270+452687
#tagged K0S meaningful mass ranges 0.434 to 0.551 'showed' bars. Less were not 'seeable'

# K0S_mcErrors for K0S isSignal that disagree with mcTruth matching
# 4.0      3738 #k0s reconstructed from secondaries/wrong hypothesis/was a k0s but not from the right (or any) pions etc\decay inflight. Might be worth keeping
# 32.0      205 #Missed a FSR massive parti
# 16.0       66 #missed photon, (higher neutral energy in ROE?)
# 136.0      39 #Missed charge and neutrino i think
# 48.0       38 #miss part and neut?
# 164.0      25
# 132.0      15
# 180.0       8
# 188.0       4
# 148.0       4
# 56.0        4
# 172.0       3
# 40.0        2
# 128.0       1
#Trk is signal/AM dont matter
#102632, 382695, 386847, 521969
#345784 577965--383010
#102560 102632
#570667 - 521969 mcPDG right. isSignal not -trk. 
#(np.int64(349514), np.int64(345784)) k0s real and tracksig. K0S and track sig. 
#382247 trk and k0s mcpdg. 
# (378139, np.int64(349514), np.int64(345784), np.int64(382247)) , k0s sig trkmc | trksig,k0smc | kos trk sig | k0s trk MC, 
#102560, 102632 k0s and trk is signal under trkk0s is sigam. First also all 3 for full recon day ignoring pi. trkK0S isSIGMA dominates
#58282 overall, 222 sig pure. 346061-345784,378444-378139 #k0s and trk isSignal vs acceptmiss (higher), then same but trkMcPDG. 
#59.83% if selecting k0s and trk is signal

"""Extra important notes"""

'Extra important notes'

In [4]:

#@njit
def get_fom_win(sig_values, bkg_values, nbins=1000): #This is a trap, never use 1000 bins, 700 'works'
    minval = min(sig_values.min(), bkg_values.min())
    maxval = max(sig_values.max(), bkg_values.max())
    thresholds = np.linspace(minval, maxval, nbins+1)
    best_fom = 0.0
    best_low = minval
    best_high = maxval
    for i in range(nbins):
        t_low = thresholds[i]
        for j in range(i+1, nbins+1):
            t_high = thresholds[j]
            S = np.sum((sig_values >= t_low) & (sig_values <= t_high))
            B = np.sum((bkg_values >= t_low) & (bkg_values <= t_high))
            if S + B > 0:
                fom = S / np.sqrt(S + B)
                if fom > best_fom:
                    best_fom = fom
                    best_low = t_low
                    best_high = t_high
    return best_low, best_high, best_fom

#@njit
def fom_thresh_low(sig_values, bkg_values, nbins=1000):
    minval = min(sig_values.min(), bkg_values.min())
    maxval = max(sig_values.max(), bkg_values.max())
    thresholds = np.linspace(minval, maxval, nbins+1)
    best_fom = 0.0
    best_thresh = thresholds[0]
    for t in thresholds:
        S = np.sum(sig_values > t)
        B = np.sum(bkg_values > t)
        if S + B > 0:
            fom = S / np.sqrt(S + B)
            if fom > best_fom:
                best_fom = fom
                best_thresh = t
    return best_thresh, best_fom

#@njit
def fom_thresh_high(sig_values, bkg_values, nbins=1000):
    minval = min(sig_values.min(), bkg_values.min())
    maxval = max(sig_values.max(), bkg_values.max())
    thresholds = np.linspace(minval, maxval, nbins+1)
    best_fom = 0.0
    best_thresh = thresholds[0]
    for t in thresholds[::-1]:
        S = np.sum(sig_values < t)
        B = np.sum(bkg_values < t)
        if S + B > 0:
            fom = S / np.sqrt(S + B)
            if fom > best_fom:
                best_fom = fom
                best_thresh = t
    return best_thresh, best_fom

def get_foms(sig_values, bkg_values, nbins=300):
    sig = np.asarray(sig_values)
    bkg = np.asarray(bkg_values)
    minval = min(sig.min(), bkg.min())
    maxval = max(sig.max(), bkg.max())

    coarse_low, coarse_high, _ = get_fom_win(sig, bkg, nbins)
    pad = (coarse_high - coarse_low) * 0.1
    lo_bnd = max(minval, coarse_low - pad)
    hi_bnd = min(maxval, coarse_high + pad)

    def neg_fom(lohi):
        low, high = lohi
        if low >= high:
            return np.inf
        S = np.sum((sig >= low) & (sig <= high))
        B = np.sum((bkg >= low) & (bkg <= high))
        if S + B == 0:
            return np.inf
        return -S / np.sqrt(S + B)
    #[(lo_bnd, hi_bnd), (lo_bnd, hi_bnd)]
    res = minimize(neg_fom, x0=[coarse_low, coarse_high], bounds=[(minval, maxval), (minval, maxval)], method='L-BFGS-B')
    low, high = res.x
    fom = -res.fun
    center = (low + high) / 2
    return low, high, center, fom
    
def get_modes_sig_Bchg(df):
    df_temp = df.copy()
    decay_sectors_chg = {
    "semileptonic": {"range": range(1001, 1081), "events": 0},
    "hadronic": {"range": list(range(1081, 1085)) + list(range(1798, 1811)), "events": 0},
    "radiative decays": {"range": range(1085, 1093), "events": 0},
    "electroweak penguin": {"range": range(1093, 1110), "events": 0}, #Possibility at least, could be wrong. Documentation misleading
    "leptonic": {"range": range(1110, 1113), "events": 0}, 
    "hadronic charmless": {"range": range(1113, 1609), "events": 0},
    "charmonium": {"range": range(1609, 1680), "events": 0},
    "charm mesons": {"range": range(1680, 1798), "events": 0},
    "baryonic": {"range": range(1811, 1830), "events": 0},
    "null": {"range": [99], "events": 0}, #possible issue if 99 daughters etc
    "daughters": {"range": range(1,1001), "events": 0}}
    df_temp['signal'] = pd.NA
    df_temp.loc[df_temp['eSL_charge'] < 0, 'signal'] = abs(df_temp.loc[df_temp['eSL_charge'] < 0, 'aBplusMode'])
    df_temp.loc[df_temp['eSL_charge'] > 0, 'signal'] = abs(df_temp.loc[df_temp['eSL_charge'] > 0, 'aBminusMode'])
    for mode, events in df_temp['signal'].value_counts().items():
        for sector, ranges in decay_sectors_chg.items():
            if mode in ranges['range']:
                decay_sectors_chg[sector]['events'] += events
                break
    rare = ["leptonic","radiative decays","electroweak penguin"]
    hadronic = ["hadronic","hadronic charmless","baryonic","charmonium","charm mesons"]
    ovw_cmc = {cat:sum(decay_sectors_chg[mode]['events'] for mode in modes) for cat, modes in { #Condense
        "semileptonic": ["semileptonic"],
        "ew/leptonic/radiative/rare": rare,
        "hadronic": ["hadronic","hadronic charmless","baryonic","charmonium","charm mesons"],  
        "unclassified": ["daughters","null"]}.items()}
    obs = len(df_temp)-ovw_cmc['unclassified']
    x = ovw_cmc['ew/leptonic/radiative/rare']
    y = ovw_cmc['hadronic']
    d_rare = {mode: decay_sectors_chg[mode]['events']/x*100 if x != 0 else 0 for mode in rare}
    d_hadro = {mode: decay_sectors_chg[mode]['events']/y*100 if y != 0 else 0 for mode in hadronic}
    ovw_cmc_2 = {cat:sum(decay_sectors_chg[mode]['events'] for mode in modes)/obs*100 if obs != 0 else 0 for cat, modes in { #Condense
        "semileptonic": ["semileptonic"],
        "ew/leptonic/radiative/rare": ["leptonic","radiative decays","electroweak penguin"],
        "hadronic": ["hadronic","hadronic charmless","baryonic","charmonium","charm mesons"]}.items()}
    ovw_cmc = {mode: ovw_cmc[mode]/len(df_temp)*100 if len(df_temp) != 0 else 0 for mode in ovw_cmc}
    return decay_sectors_chg,ovw_cmc,ovw_cmc_2, d_rare,d_hadro

def set_modes_sig_Bchg(df):
    df_t = df.copy()
    df_t['signal'] = pd.NA
    df_t.loc[df_t['eSL_charge'] < 0, 'signal'] = abs(df_t.loc[df_t['eSL_charge'] < 0, 'aBplusMode'])
    df_t.loc[df_t['eSL_charge'] > 0, 'signal'] = abs(df_t.loc[df_t['eSL_charge'] > 0, 'aBminusMode'])
    return df_t

In [5]:
def get_mode_dicts(filepath = ''): # Ordered keys as: Bplus, Bminus, B0, Bbar0, Bs0, Bsbar0, Dstplus, Dstminus, Dsplus, Dsminus, Dplus, Dminus, D0, Dbar0, Tauplus, Tauminus

    #https://docs.belle2.org/files/541/BELLE2-NOTE-TE-2021-002/1/BELLE2-NOTE-TE-2021-002.pdf referenced from MC gen tag tool in BASF2, tex files from MC gen Tag tool
    #basf2/analysis/utility/src/GenBsTag.cc
    """Note: This is one way to scan and get the corresponding decay files from the published tool relating to the internal note on gitlab. 
    However, one could in theory do the same thing on the files in BASF2. 

    To do this, goto Basf2 in gitlab. Then analysis/utility/. The tool GENMCTAG is mostly in the src files as GenBplusTag.cc
    Going through each raw file with a scan or by calling the methods and getting the corresponding decay and string is another way. 
    Will implement if time, along with a reverse look up. Ie, input decay string, get ID.
    if (GenBsTag::PcheckDecay(genpart, -10431, -15, 16)) {
    return +1 * (100000 * m_nPhotos + 1016); is how its done in the basf2 implementation. 
    So, going through the list iteratively as how they've designed it with a PDG hash table of the particles would be 
    rather simple for a next and more robust iteration. Specifically incase the tool is updated in basf2 only but not in the publication files on gitlab. 
    
    For now, I've attached a simple method to demonstrate using this function to show decay strings
    Originally this was done using a bunch of dictionaries but I swapped to a nested version where you can specify the mother rather than the index.
    I also included the mother and an arrow in the decay string, but left in the commented version without that which is more concise and implies the mother 
    in the correct context.
    # update: I made it more concise and read through a CSV I generated with the old method - Downside is would need updated with some changes if more added. 
    """
    
    master_dict = {
    'Bplus': {},'Bminus': {},
    'B0': {},'Bbar0': {},
    'Bs0': {},'Bsbar0': {},
    'Dstplus': {},'Dstminus': {},
    'Dsplus': {},'Dsminus': {},
    'Dplus': {},'Dminus': {},
    'D0': {},'Dbar0': {},
    'Tauplus': {},'Tauminus': {}}
    with open(filepath+'all_decays.csv',encoding='utf-8') as fh:
            for line in fh:
                line = line.strip()
                if 'Decay ID' in line:
                    key = line.split(',')[0]
                    continue
                mode, modeid = line.split(',')
                master_dict[key][int(modeid)] = mode


    return master_dict



In [6]:
def grab_signals_Bchg(df):
    df_t = df.copy()
    df_t['signal'] = pd.NA
    sig_bp = abs(df_t.loc[df_t['eSL_charge'] < 0, 'aBplusMode'])
    sig_bm = abs(df_t.loc[df_t['eSL_charge'] > 0, 'aBminusMode'])
    return sig_bp, sig_bm

In [7]:
def initial_sel1(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.93')
    df_t = df_t.query('Eextra_ROE_loose < 1.84')
    df_t = df_t.query('M_ROE_loose < 6.58')
    df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    df_t = df_t.query('(K0S_InvM >= 0.4885) & (K0S_InvM <= 0.50565)') 
    df_t = df_t.query('(K0S_flightDistance > -1)')
    df_t = df_t.query('(K0S_flightDistanceErr > 0.5)')
    return df_t

def initial_sel2(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.93')
    # df_t = df_t.query('Eextra_ROE_loose < 1.84')
    # df_t = df_t.query('M_ROE_loose < 6.58')
    # df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    # df_t = df_t.query('(K0S_InvM >= 0.4885) & (K0S_InvM <= 0.50565)') 
    # df_t = df_t.query('(K0S_flightDistance > -1)')
    # df_t = df_t.query('(K0S_flightDistanceErr > 0.5)')
    return df_t

def initial_sel3(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.93')
    df_t = df_t.query('Eextra_ROE_loose < 1.84')
    df_t = df_t.query('M_ROE_loose < 6.58')
    df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    # df_t = df_t.query('(K0S_InvM >= 0.4885) & (K0S_InvM <= 0.50565)') 
    # df_t = df_t.query('(K0S_flightDistance > -1)')
    # df_t = df_t.query('(K0S_flightDistanceErr > 0.5)')
    return df_t

def initial_sel4(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.93')
    # df_t = df_t.query('Eextra_ROE_loose < 1.84')
    # df_t = df_t.query('M_ROE_loose < 6.58')
    # df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    df_t = df_t.query('(K0S_InvM >= 0.4885) & (K0S_InvM <= 0.50565)') 
    df_t = df_t.query('(K0S_flightDistance > -1)')
    df_t = df_t.query('(K0S_flightDistanceErr > 0.5)')
    return df_t
    
def grab_signal_D(df):
    df_t = df.copy()
    mask_p = df_t['eSL_charge'] < 0
    sig_Dbar0 = abs(df_t.loc[mask_p, 'aDbar0Mode'])
    sig_Dminus = abs(df_t.loc[mask_p, 'aDminusMode'])
    mask_m = df_t['eSL_charge'] > 0
    sig_D0 = abs(df_t.loc[mask_m, 'aD0Mode'])
    sig_Dplus = abs(df_t.loc[mask_m, 'aDplusMode'])
    return sig_Dbar0, sig_Dminus, sig_D0, sig_Dplus


In [8]:
def get_modes_sig_Bchg2(df):
    df_temp = df.copy()
    dau_modes = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16]
    decay_sectors_chg = {
    "semileptonic": {"range": range(1001, 1081), "events": 0},
    "hadronic": {"range": list(range(1081, 1085)) + list(range(1798, 1811)), "events": 0},
    "radiative decays": {"range": range(1085, 1093), "events": 0},
    "electroweak penguin": {"range": range(1093, 1110), "events": 0}, #Possibility at least, could be wrong. Documentation misleading
    "leptonic": {"range": range(1110, 1113), "events": 0}, 
    "hadronic charmless": {"range": range(1113, 1609), "events": 0},
    "charmonium": {"range": range(1609, 1680), "events": 0},
    "charm mesons": {"range": range(1680, 1798), "events": 0},
    "baryonic": {"range": range(1811, 1830), "events": 0},
    "null": {"range": [99], "events": 0}, #possible issue if 99 daughters etc
    "daughters": {"range": dau_modes, "events": 0}}
    df_temp['signal'] = pd.NA
    df_temp.loc[df_temp['eSL_charge'] < 0, 'signal'] = abs(df_temp.loc[df_temp['eSL_charge'] < 0, 'aBplusMode'])
    df_temp.loc[df_temp['eSL_charge'] > 0, 'signal'] = abs(df_temp.loc[df_temp['eSL_charge'] > 0, 'aBminusMode'])
    for mode, events in df_temp['signal'].value_counts().items():
        for sector, ranges in decay_sectors_chg.items():
            if mode in ranges['range']:
                decay_sectors_chg[sector]['events'] += events
                break
    rare = ["leptonic","radiative decays","electroweak penguin"]
    hadronic = ["hadronic","hadronic charmless","baryonic","charmonium","charm mesons"]
    ovw_cmc = {cat:sum(decay_sectors_chg[mode]['events'] for mode in modes) for cat, modes in { #Condense
        "semileptonic": ["semileptonic"],
        "ew/leptonic/radiative/rare": rare,
        "hadronic": ["hadronic","hadronic charmless","baryonic","charmonium","charm mesons"],  
        "unclassified": ["daughters","null"]}.items()}
    obs = len(df_temp)-ovw_cmc['unclassified']
    x = ovw_cmc['ew/leptonic/radiative/rare']
    y = ovw_cmc['hadronic']
    z = ovw_cmc['unclassified']
    d_rare = {mode: decay_sectors_chg[mode]['events']/x*100 if x != 0 else 0 for mode in rare}
    d_hadro = {mode: decay_sectors_chg[mode]['events']/y*100 if y != 0 else 0 for mode in hadronic}
    d_uncl = {mode: df_temp.loc[df_temp['signal'] == mode].shape[0]/z*100 if z != 0 else 0 for mode in dau_modes} #Different since different mode type, actual IDs instead
    ovw_cmc = {mode: ovw_cmc[mode]/len(df_temp)*100 if len(df_temp) != 0 else 0 for mode in ovw_cmc}
    return decay_sectors_chg,ovw_cmc, d_rare,d_hadro, d_uncl

In [9]:
#{'particle': 'key', ['colors'], 'label'}

def plot_by_trk(mclist, dat, func, key_dict, filepath): #Change key to dict with nice info.. Use dict to keep param small with all info
    mini = key_dict['particle'][3]
    maxi = key_dict['particle'][4]
    bins = key_dict['particle'][5]
    key = key_dict['particle'][0]
    type_cols = key_dict['particle'][1]
    p_lab = key_dict['particle'][2]
    type_labs = df_namesmc

    def make_fig():
        fig = plt.figure(figsize=(30, 25), dpi=300, constrained_layout=True) #Check res later
        gridspc = fig.add_gridspec(4, 6, height_ratios=[4, 1, 4, 1])
        axms = []
        axrs = []
        for row, start, end in [(0, 0, 2), (0, 2, 4), (0, 4, 6), (2, 1, 3), (2, 3, 5)]:
            axm = fig.add_subplot(gridspc[row, start:end])
            axr = fig.add_subplot(gridspc[row+1, start:end], sharex=axm)
            axms.append(axm)
            axrs.append(axr)
        for ax in axms:
            ax.tick_params(labelbottom=False)
        return fig, axms, axrs

    sim_trks = [[sel(mc) for mc in mclist] for sel in sel_funcs] #Decompose into trk->gentype->df
    dat_trks = [sel(dat) for sel in sel_funcs] #trk->df
    sim_hists = [[bh.Histogram(bh.axis.Regular(bins, mini, maxi)).fill(sim[key], weight=np.full_like(sim[key], 0.25))
     for sim in sims] for sims in sim_trks]
    dat_hists = [bh.Histogram(bh.axis.Regular(bins, mini, maxi)).fill(dat_sel[key]) for dat_sel in dat_trks]

    sim_trks2 = [[sel(func(mc)) for mc in mclist] for sel in sel_funcs] #Decompose into trk->gentype->df
    dat_trks2 = [sel(func(dat)) for sel in sel_funcs] #trk->df
    sim_hists2 = [[bh.Histogram(bh.axis.Regular(bins, mini, maxi)).fill(sim[key], weight=np.full_like(sim[key], 0.25))
     for sim in sims] for sims in sim_trks2]
    dat_hists2 = [bh.Histogram(bh.axis.Regular(bins, mini, maxi)).fill(dat_sel[key]) for dat_sel in dat_trks2]

    eff_sim_total = [sum(sum(gentype.view() for gentype in sim_hist)) for sim_hist in sim_hists]
    eff_dat_total = [sum(dat_hist.view()) for dat_hist in dat_hists]

    for dat_hists_cur, sim_hists_cur, sel_text in [
        (dat_hists, sim_hists, 'No_Selections'),
        (dat_hists2, sim_hists2, 'Selections')
    ]:
        fig, axms, axrs = make_fig()
        for i, (dat_hist, sim_hist_types, axm, axr, typ) in enumerate(zip(dat_hists_cur, sim_hists_cur, axms, axrs,
                                                                         [rf'\mu^\pm',rf'e^\pm',rf'\pi^\pm',rf'K^\pm', rf'All Modes Combined'])):
            bin_edges = dat_hist.axes[0].edges
            bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
            bin_widths = np.diff(bin_edges)
            bottom = np.zeros_like(bin_centers)
            for gentypehist, col, lab in zip(sim_hist_types, type_cols, type_labs):
                counts = gentypehist.view()
                axm.bar(bin_centers, counts, width=bin_widths, bottom=bottom,
                        color=col, align='center', edgecolor='black', label=lab)
                bottom += counts

            axm.errorbar(bin_centers, dat_hist.view(), yerr=np.sqrt(dat_hist.view()),
                         fmt='o', color='black', label='Data', markersize=3)
            axm.set_xlim(mini, maxi)
            axm.set_ylabel(rf"Candidates / ({bin_widths[0]:.5f} GeV/$c^2$)")
            title = rf"Data and MC Comparison for ${typ}$ Channel with ${sel_text}$"
            axm.set_title(title)
            axm.text(
                0.02, 0.99,
                r"$\mathbf{Belle\,II}$ 2025 Preliminary"+'\n'+
                r"$\int \mathcal{L}\,dt = 365.290\,\mathrm{fb}^{-1}$",
                fontsize=12,
                verticalalignment='top',
                horizontalalignment='left',
                transform=axm.transAxes,
                bbox=dict(facecolor='white', alpha=0.0, edgecolor='none', pad=2))
            if sel_text == 'Selections':
                eff_sim_sel = sum(bottom) / eff_sim_total[i] if eff_sim_total[i] != 0 else 0
                eff_dat_sel = sum(dat_hist.view()) / eff_dat_total[i] if eff_dat_total[i] != 0 else 0
                axm.text(
                    0.98, 0.85,
                    rf"$\mathrm{{Eff_{{Data}}\%}} \: {eff_dat_sel*100:.3f}$\%" + '\n' +
                    rf"$\mathrm{{Eff_{{Sim}}\%}}: \: {eff_sim_sel*100:.3f}$\%" + '\n' + 
                    f'N Sim: {sum(bottom)}' + '\n' + f'N Dat: {sum(dat_hist.view())}',
                    fontsize=12,
                    verticalalignment='top',
                    horizontalalignment='right',
                    transform=axm.transAxes,
                    bbox=dict(facecolor='white', alpha=0.0, edgecolor='none', pad=2))
            else:
                axm.text(
                    0.98, 0.85, 
                    f'N Sim: {sum(bottom)}' + '\n' + f'N Dat: {sum(dat_hist.view())}',
                    fontsize=12,
                    verticalalignment='top',
                    horizontalalignment='right',
                    transform=axm.transAxes,
                    bbox=dict(facecolor='white', alpha=0.0, edgecolor='none', pad=2))
            with np.errstate(divide='ignore', invalid='ignore'):
                ratio = np.where(bottom > 0, dat_hist.view() / bottom, np.nan)
                ratio_err = np.where(bottom > 0, np.sqrt(dat_hist.view()) / bottom, 0)
            axm.bar(bin_centers, bottom=bottom - np.sqrt(bottom), height=2 * np.sqrt(bottom), width=bin_widths[0],
                    hatch='////', edgecolor="dimgrey", fill=False, lw=0, label='Simulation Uncertainty')
            axr.errorbar(bin_centers, ratio, yerr=ratio_err, fmt='o', color='black', markersize=3)
            axr.set_ylim(0, 2)
            axr.axhline(1, color='black', linestyle='--')
            axr.set_ylabel(rf"$\frac{{\mathrm{{Data}}}}{{\mathrm{{Simulation}}}}$")
            axr.set_xlabel(rf"${p_lab}\:[{{\rm GeV}}/c^2]$")
            axr.bar(bin_centers,
                    bottom=np.nan_to_num(1 - ratio_err, nan=2), height=np.nan_to_num(2 * ratio_err, nan=2), width=bin_widths[0],
                    edgecolor="dimgrey", hatch="////", fill=False, lw=0)
            axm.legend()
        fig.savefig(f'{filepath}plot_by_trk_{sel_text}_{datetime.now():%Y%m%d_%H%M%S}.svg', format='svg')
        fig.savefig(f'{filepath}plot_by_trk_{sel_text}_{datetime.now():%Y%m%d_%H%M%S}.pdf', format='pdf')
        plt.show()


def plot_full_ovc_summary(ovc_lis, df, modes, filepath, count=50):
    ovc, decomp_rare, decomp_hadron, decomp_unclas = ovc_lis
    fig = plt.figure(figsize=(30, 28))
    grid = fig.add_gridspec(6, 2)

    labels_all = list(ovc.keys())
    sizes_all = list(ovc.values())
    explode_all = [0.15 if s < 5 else 0 for s in sizes_all]

    ax1 = fig.add_subplot(grid[0, :])
    ax1.pie(
        sizes_all,
        labels=labels_all,
        explode=explode_all,
        autopct=lambda pct: f"{pct:.3f}%" if pct > 0 else "",
        startangle=140,
        pctdistance=0.85,
        labeldistance=1.15,
        textprops={'fontsize': 11},
        wedgeprops={'linewidth': 1, 'edgecolor': 'white'}
    )
    ax1.set_title("Charged B Decays: All Categories", fontsize=16)

    rare_items = sorted(decomp_rare.items(), key=lambda x: x[1])
    rare_labels, rare_vals = zip(*rare_items)

    ax2 = fig.add_subplot(grid[1, 0])
    ax2.barh(rare_labels, rare_vals, color='skyblue', edgecolor='black')
    ax2.set_title("Charged B Decays: Breakdown of EW/Leptonic/Radiative/Rare", fontsize=14)
    ax2.set_xlabel("Percentage of All Rare Decays (%)")
    for j, v in enumerate(rare_vals):
        ax2.text(v+0.2, j, f"{v:.3f}%", va='center', fontsize=10)

    hadron_items = sorted(decomp_hadron.items(), key=lambda x: x[1])
    hadron_labels, hadron_vals = zip(*hadron_items)

    ax3 = fig.add_subplot(grid[1, 1])
    ax3.barh(hadron_labels, hadron_vals, color='lightgreen', edgecolor='black')
    ax3.set_title("Charged B Decays: Breakdown of Hadronic", fontsize=14)
    ax3.set_xlabel("Percentage of All Hadronic Decays (%)")
    for j, v in enumerate(hadron_vals):
        ax3.text(v+0.2, j, f"{v:.3f}%", va='center', fontsize=10)

    unclas_items = sorted(decomp_unclas.items(), key=lambda x: x[1])
    unclas_labels, unclas_vals = zip(*unclas_items)
    
    ax4 = fig.add_subplot(grid[2, 0])
    y_pos = range(len(unclas_labels))
    ax4.barh(y_pos, unclas_vals, color='lightgrey', edgecolor='black')
    ax4.set_title("Charged B Decays: Breakdown of Unclassified Modes", fontsize=14)
    ax4.set_xlabel("Percentage of All Unclassified Decays (%)")
    ax4.set_yticks(y_pos)
    ax4.set_yticklabels(unclas_labels)
    for j, v in enumerate(unclas_vals):
        ax4.text(v + 0.2, j, f"{v:.3f}%", va='center', fontsize=10)

    bp, bm = grab_signals_Bchg(df_asim)
    top_bp = bp[bp != 99.0].value_counts().head(count)
    top_bm = bm[bm != 99.0].value_counts().head(count)
    x = set(top_bp.index)
    y = set(top_bm.index)
    xy = list(x - y)
    yx = list(y - x)
    combined = top_bp.add(top_bm, fill_value=0).astype(int)
    combined = combined.sort_index()
    cols = combined.index
    vals = combined.values

    for val0 in yx:
        top_bp[val0] = 0
    for val0 in xy:
        top_bm[val0] = 0

    top_bp = top_bp.sort_index()
    top_bm = top_bm.sort_index()
    bpl = r'$B^+$'
    bmin = r'$B^-$'
    labs = [f"${all_modes['Bplus'][col]}$" for col in cols]
    ax5 = fig.add_subplot(grid[3, :])
    ax5.barh(range(len(vals)), top_bm.values + top_bp.values, color='black', edgecolor='black', label=bmin, zorder=0)
    ax5.barh(range(len(vals)), top_bp.values, color='limegreen', edgecolor='black', label=bpl)
    ax5.set_xlabel("Candidates / (Decay Mode)", fontsize=12)
    ax5.set_ylabel("Decay Mode (B+ for simplification)", fontsize=12)
    ax5.set_title(f"Top {count} Decay Modes for Signal Side {bpl} and {bmin} Candidates Stacked")
    ax5.set_yticks(range(len(cols)))
    ax5.set_yticklabels(labs, fontsize=10)
    ax5.grid(axis='x', linestyle='--', alpha=0.7)
    ax5.legend()

    Dbar0, Dminus, D0, Dplus = grab_signal_D(df_asim)

    for idx, (D1, D2, label, type1, type2) in enumerate([
        (Dbar0, D0, "Dbar0 vs D0",r'$\bar{D}^0$', r'$D^0$'),
        (Dplus, Dminus, "D$^+$ vs D-", r'$D^+$', r'$D^-$')]):
        top_D1 = D1[D1 != 99.0].value_counts().head(count)
        top_D2 = D2[D2 != 99.0].value_counts().head(count)
        x = set(top_D1.index)
        y = set(top_D2.index)
        xy = list(x - y)
        yx = list(y - x)
        combined = top_D1.add(top_D2, fill_value=0).astype(int)
        combined = combined.sort_index()
        cols = combined.index
        vals = combined.values

        for val0 in yx:
            top_D1[val0] = 0
        for val0 in xy:
            top_D2[val0] = 0

        top_D1 = top_D1.sort_index()
        top_D2 = top_D2.sort_index()

        mode_map = {
            'aDbar0Mode': 'Dbar0',
            'aD0Mode': 'D0',
            'aDplusMode': 'Dplus',
            'aDminusMode': 'Dminus'}
        labs = [f"${all_modes[mode_map[D1.name]][col]}$" for col in cols]
        ax = fig.add_subplot(grid[4 + idx, :])
        ax.barh(range(len(vals)), top_D2.values + top_D1.values, color='black', edgecolor='black', label=type2, zorder=0)
        ax.barh(range(len(vals)), top_D1.values, color='deepskyblue', edgecolor='black', label=type1)
        ax.set_xlabel("Candidates / (Decay Mode)", fontsize=12)
        ax.set_ylabel(f"Decay Mode ({type1} for simplification)", fontsize=12)
        ax.set_title(f"Top {count} Decay Modes for Signal Side {type1} and {type2} Candidates Stacked")
        ax.set_yticks(range(len(cols)))
        ax.set_yticklabels(labs, fontsize=10)
        ax.grid(axis='x', linestyle='--', alpha=0.7)
        ax.legend()
    plt.tight_layout()
    fig.savefig(f'{filepath}plot_full_ovc_summary.svg', format='svg')
    fig.savefig(f'{filepath}plot_full_ovc_summary.pdf', format='pdf')
    plt.show()

In [10]:
def selec_1(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.8')
    #df_t = df_t.query('Eextra_ROE_loose < 1.84')
    #df_t = df_t.query('M_ROE_loose < 6.58')
    #df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    df_t = df_t.query('(K0S_InvM >= 0.485) & (K0S_InvM <= 0.51)') 
    df_t = df_t.query('(K0S_flightDistance > -1)')
    df_t = df_t.query('(K0S_flightDistanceErr < 0.5)')
    return df_t

def selec_2(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.94')
    #df_t = df_t.query('Eextra_ROE_loose < 1.84')
    #df_t = df_t.query('M_ROE_loose < 6.58')
    #df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.7) & (trk_pCM < 2.7) & (K0S_pCM < 2.7)')
    df_t = df_t.query('(K0S_InvM >= 0.48) & (K0S_InvM <= 0.515)') 
    df_t = df_t.query('(K0S_flightDistance > -1)')
    #df_t = df_t.query('(K0S_flightDistanceErr < 0.5)')
    return df_t

def selec_3(df):
    df_t = df.copy()
    df_t = df_t.query('nROE_Ch == 0')
    df_t = df_t.query('Eextra_ROE < 0.8')
    #df_t = df_t.query('Eextra_ROE_loose < 1.84')
    #df_t = df_t.query('M_ROE_loose < 6.58')
    #df_t = df_t.query('M_ROE < 4')
    df_t = df_t.query('BSL_BchiProb > 1e-8')
    df_t = df_t.query('(BSL_cosBY >= -3) & (BSL_cosBY <= 1.1)')
    df_t = df_t.query('(eSL_pCM < 2.5) & (trk_pCM < 2.5) & (K0S_pCM < 2.5)')
    df_t = df_t.query('(K0S_InvM >= 0.485) & (K0S_InvM <= 0.51)') 
    df_t = df_t.query('(K0S_flightDistance > -1)')
    df_t = df_t.query('(K0S_flightDistanceErr < 0.5)')
    return df_t